In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/sample_submission.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/item_meta.jsonl
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/test.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train_reviews.jsonl
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k128_iter100_lr0.005_reg0.01_seed42_val.csv
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k128_iter100_lr0.01_reg0.001_seed42_val.csv
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k128_iter100_lr0.01_reg0.01_seed42_val.csv
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k128_iter100_lr0.01_reg0.05_seed42_val.csv
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k128_iter100_lr0.05_reg0.01_seed42_val.csv
/kaggle/input/datasets/kimeoryeong/11-11-backup/BPRMF_k32_iter100_lr0.01_reg

In [2]:
# 셀 0: 라이브러리 설치
import socket
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

TARGET = "0.7.2"

try:
    installed = version("implicit")
except PackageNotFoundError:
    installed = None

if installed != TARGET:
    supported_tags = set(sys_tags())
    wheels = []

    for path in Path("/kaggle/input").rglob(f"implicit-{TARGET}-*.whl"):
        _, _, _, wheel_tags = parse_wheel_filename(path.name)
        if supported_tags.intersection(wheel_tags):
            wheels.append(path)

    pip = [sys.executable, "-m", "pip", "install"]

    if wheels:
        wheel = sorted(wheels)[0]
        print(f"로컬 wheel 사용 → {wheel.name}")
        subprocess.run(
            [*pip, "--no-index", "--find-links", str(wheel.parent), str(wheel)],
            check=True,
        )
    else:
        try:
            socket.getaddrinfo("pypi.org", 443, type=socket.SOCK_STREAM)
        except socket.gaierror:
            raise RuntimeError(
                "PyPI 주소를 조회할 수 없습니다. "
                "Kaggle Settings에서 Internet을 켜고 셀 0을 다시 실행하세요. "
                "Internet을 켤 수 없다면 현재 Python/Linux 환경과 호환되는 "
                "implicit 0.7.2 wheel을 Add Input으로 연결하세요."
            ) from None

        subprocess.run(
            [*pip, "--retries", "0", "--timeout", "15", f"implicit=={TARGET}"],
            check=True,
        )

# 실제 ALS import까지 확인한 뒤 다음 셀로 진행
from implicit.cpu.als import AlternatingLeastSquares

print(f"implicit {version('implicit')} · CPU ALS 준비 완료")

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  error: subprocess-exited-with-error
  
  × Building wheel for implicit (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for implicit
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> implicit


Failed to build implicit


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '--retries', '0', '--timeout', '15', 'implicit==0.7.2']' returned non-zero exit status 1.

### 셀 1. 라이브러리와 입력 경로 설정

In [ ]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from threadpoolctl import threadpool_limits

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

competition_dirs = sorted({
    path.parent for path in INPUT.rglob("train.csv")
    if (path.parent / "test.csv").is_file()
})

if len(competition_dirs) != 1:
    raise RuntimeError(
        f"train.csv와 test.csv가 함께 있는 대회 입력을 하나 연결하세요: "
        f"{competition_dirs}"
    )

TRAIN = competition_dirs[0] / "train.csv"
TEST = competition_dirs[0] / "test.csv"

kit_scripts = sorted(INPUT.rglob("teamkit.py"))

if not kit_scripts:
    kit_zips = sorted(INPUT.rglob("amazon_topk_teamkit_20261110.zip"))
    if len(kit_zips) != 1:
        raise RuntimeError("팀 키트 ZIP을 Private Dataset으로 연결하세요.")

    kit_dir = WORK / "teamkit_input"
    with zipfile.ZipFile(kit_zips[0]) as archive:
        archive.extractall(kit_dir)
    kit_scripts = sorted(kit_dir.rglob("teamkit.py"))

if len(kit_scripts) != 1:
    raise RuntimeError(f"팀 키트 입력을 하나 연결하세요: {kit_scripts}")

TEAMKIT = kit_scripts[0]
VAL = TEAMKIT.parent / "validation"

print("대회 데이터:", TRAIN.parent)
print("팀 키트:", TEAMKIT.parent)

### 2. 팀 공통 검증 데이터 확인

In [ ]:
def run_teamkit(command, **paths):
    args = [sys.executable, str(TEAMKIT), command]
    for name, value in paths.items():
        args.extend([f"--{name.replace('_', '-')}", str(value)])

    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    return json.loads(result.stdout)

split = run_teamkit("verify-split", dir=VAL)
manifest = json.loads((VAL / "manifest.json").read_text(encoding="utf-8"))

if split["split_id"] != "f0dfad2f4431a876":
    raise ValueError(f"팀 공통 분할 ID와 다릅니다: {split['split_id']}")

train_hash = hashlib.sha256(TRAIN.read_bytes()).hexdigest()
if train_hash != manifest["source_train_sha256"]:
    raise ValueError("대회 입력의 train.csv가 팀 공통 분할의 원본과 다릅니다.")

sys.path.insert(0, str(TEAMKIT.parent))
from teamkit import read_interactions, read_users, top20_unseen, write_predictions

print(split)

### 셀 3 · ALS 학습과 추천 함수
첫 설정은 64차원, 정규화 0.05, 관측 신뢰도 1, 20회 반복, 시드 42예요. 사용자와 상품의 상호작용은 중복을 제거해 1로 기록해요. 이후 alpha를 높이면 관측된 상호작용에 더 높은 신뢰도를 부여해요. implicit 0.7.2의 fit에는 사용자 × 상품 CSR 행렬을 넘겨요

In [ ]:
#from implicit.cpu.als import AlternatingLeastSquares

SEED = 42
ITERATIONS = 20
THREADS = 2
BATCH_SIZE = 256
TOP_K = 20

BASE = {"factors": 64, "regularization": 0.05, "alpha": 1.0}

VAL_FIT = VAL / "val_train.csv"
VAL_USERS = VAL / "val_users.csv"


def load_fit_data(fit_path):
    fit = pd.read_csv(
        fit_path,
        usecols=["user_id", "item_id"],
        dtype={"user_id": str, "item_id": str},
    )
    user_rows, user_ids = pd.factorize(fit["user_id"], sort=True)
    item_cols, item_ids = pd.factorize(fit["item_id"], sort=True)

    matrix = csr_matrix(
        (np.ones(len(fit), dtype=np.float32), (user_rows, item_cols)),
        shape=(len(user_ids), len(item_ids)),
    )
    matrix.data.fill(1.0)
    matrix.sort_indices()

    seen, popularity = read_interactions(fit_path)
    catalog = set(popularity)
    fallback = sorted(catalog, key=lambda item: (-popularity[item], int(item)))

    return (
        matrix,
        {user: row for row, user in enumerate(user_ids)},
        np.asarray(item_ids),
        seen,
        fallback,
        catalog,
    )


def predict_als(fit_data, users_path, output_path, config):
    matrix, row_by_user, item_ids, seen, fallback, catalog = fit_data
    users = read_users(users_path)
    predictions = {}

    model = AlternatingLeastSquares(
        **config,
        iterations=ITERATIONS,
        random_state=SEED,
        num_threads=THREADS,
    )

    with threadpool_limits(limits=1, user_api="blas"):
        model.fit(matrix)

        for start in range(0, len(users), BATCH_SIZE):
            batch = users[start:start + BATCH_SIZE]
            known = [user for user in batch if user in row_by_user]

            if known:
                rows = np.asarray([row_by_user[user] for user in known])
                ranked_cols, scores = model.recommend(
                    rows,
                    matrix[rows],
                    N=min(TOP_K, matrix.shape[1]),
                    filter_already_liked_items=True,
                )
                for user, cols, row_scores in zip(known, ranked_cols, scores):
                    valid = (cols >= 0) & (cols < len(item_ids))
                    valid &= np.isfinite(row_scores)
                    predictions[user] = top20_unseen(
                        user, item_ids[cols[valid]], seen, fallback, catalog
                    )

            for user in batch:
                if user not in row_by_user:
                    predictions[user] = top20_unseen(
                        user, (), seen, fallback, catalog
                    )

    write_predictions(output_path, users, predictions)
    print(f"추천 저장 → {output_path.name} ({len(users):,}명)")

In [ ]:
VAL_DATA = load_fit_data(VAL_FIT)

### 셀 4 · 기본 설정 실행 및 팀 공통 양식 저장
처음에는 여기까지만 실행해서 설치, 학습, 채점이 모두 되는지 확인하면 돼요. 각 설정을 채점한 직후 결과를 als_experiments.csv에 기록해요. 같은 세션에서 실험 셀을 다시 실행하면 이미 채점한 설정은 재사용해요.

In [ ]:
VAL_DATA = load_fit_data(VAL_FIT)
EXPERIMENTS = WORK / "als_experiments.csv"
records = {}

SVD_RECALL = 0.024584
SVD_HITS = 3369


def evaluate(config):
    name = (
        f"ImplicitALS_k{config['factors']}"
        f"_reg{config['regularization']:g}"
        f"_alpha{config['alpha']:g}"
    )
    if name in records:
        return records[name]

    pred_path = WORK / f"{name}_seed{SEED}_val.csv"
    predict_als(VAL_DATA, VAL_USERS, pred_path, config)

    score = run_teamkit(
        "score", pred=pred_path, fit=VAL_FIT, truth=VAL / "val_truth.csv"
    )
    records[name] = {
        "split_id": split["split_id"],
        "model": name,
        "model_seed": SEED,
        "training_interactions": "validation/val_train.csv",
        "extra_inputs": "none",
        "recall_at_20": score["recall_at_20"],
        "hits": score["hits"],
        "denominator": score["denominator"],
        "notes": f"binary interactions; iterations={ITERATIONS}; {config}",
    }
    pd.DataFrame(records.values()).to_csv(EXPERIMENTS, index=False)
    return records[name]


def compare_results():
    table = pd.DataFrame([
        *records.values(),
        {"model": "TruncatedSVD_k32_binary", "recall_at_20": SVD_RECALL,
         "hits": SVD_HITS},
        {"model": "Popularity", "recall_at_20": 0.02283988, "hits": 3130},
    ])
    table["recall_vs_svd"] = (table["recall_at_20"] - SVD_RECALL).round(6)
    table["hits_vs_svd"] = table["hits"] - SVD_HITS
    display(
        table.sort_values("recall_at_20", ascending=False)[
            ["model", "recall_at_20", "hits", "recall_vs_svd", "hits_vs_svd"]
        ]
    )


evaluate(BASE)
compare_results()

### 셀 5 · 한 번에 한 가지씩 변경
           순서	바꾸는 값	고정하는 값
1. 잠재 차원	32, 64, 128	     /   정규화 0.05, 신뢰도 1
2. 정규화	0.01, 0.05, 0.10	/  1단계 최고 차원, 신뢰도 1
3. 관측 신뢰도	1, 10, 40	   /    1단계 최고 차원, 2단계 최고 정규화

In [ ]:
def sweep(current, parameter, values):
    candidates = [current] + [
        {**current, parameter: value}
        for value in values
        if value != current[parameter]
    ]
    scored = [(config, evaluate(config)) for config in candidates]
    winner, result = max(scored, key=lambda pair: pair[1]["hits"])
    print(
        f"{parameter} → {winner[parameter]} | "
        f"Recall@20 {result['recall_at_20']:.6f} | "
        f"적중 {result['hits']:,}건"
    )
    return winner


best_rank = sweep(BASE, "factors", [32, 64, 128])
best_reg = sweep(best_rank, "regularization", [0.01, 0.05, 0.10])
best_als = sweep(best_reg, "alpha", [1.0, 10.0, 40.0])

compare_results()
print("ALS 최고 설정 →", best_als)
print("팀 공통 실험 기록 →", EXPERIMENTS)

## Implicit ALS 최고 성능 : 0.034625

------------------------------------------------------------

## 셀 2-1 · BPR-MF 학습 및 추천 함수
기본 설정은 32차원, 100회, 학습률 0.01, 정규화 0.01로 잡았어요. BPR은 관측 상품과 미관측 상품의 순위를 비교하며 학습하므로, ALS의 alpha는 이 설정에 들어가지 않아요.

In [ ]:
from implicit.cpu.bpr import BayesianPersonalizedRanking

BPR_BASE = {
    "factors": 32,
    "iterations": 100,
    "learning_rate": 0.01,
    "regularization": 0.01,
}


def predict_bpr(fit_data, users_path, output_path, config):
    matrix, row_by_user, item_ids, seen, fallback, catalog = fit_data
    users = read_users(users_path)
    predictions = {}

    model = BayesianPersonalizedRanking(
        **config,
        random_state=SEED,
        num_threads=THREADS,
        verify_negative_samples=True,
    )

    with threadpool_limits(limits=1, user_api="blas"):
        model.fit(matrix)

        for start in range(0, len(users), BATCH_SIZE):
            batch = users[start:start + BATCH_SIZE]
            known = [user for user in batch if user in row_by_user]

            if known:
                rows = np.asarray(
                    [row_by_user[user] for user in known], dtype=np.int64
                )
                ranked_cols, scores = model.recommend(
                    rows,
                    matrix[rows],
                    N=min(TOP_K, matrix.shape[1]),
                    filter_already_liked_items=True,
                )

                for user, cols, row_scores in zip(known, ranked_cols, scores):
                    valid = (cols >= 0) & (cols < len(item_ids))
                    valid &= np.isfinite(row_scores)
                    predictions[user] = top20_unseen(
                        user, item_ids[cols[valid]], seen, fallback, catalog
                    )

            for user in batch:
                if user not in row_by_user:
                    predictions[user] = top20_unseen(
                        user, (), seen, fallback, catalog
                    )

    write_predictions(output_path, users, predictions)
    print(f"추천 저장 → {output_path.name} ({len(users):,}명)")

In [ ]:
#셀 2 · 기본 설정 학습, 채점, 결과 저장
BPR_EXPERIMENTS = WORK / "bpr_experiments.csv"
bpr_records = {}


def evaluate_bpr(config):
    name = (
        f"BPRMF_k{config['factors']}"
        f"_iter{config['iterations']}"
        f"_lr{config['learning_rate']:g}"
        f"_reg{config['regularization']:g}"
    )
    if name in bpr_records:
        return bpr_records[name]

    pred_path = WORK / f"{name}_seed{SEED}_val.csv"
    print(f"검증 실행 → {name}")
    predict_bpr(VAL_DATA, VAL_USERS, pred_path, config)

    score = run_teamkit(
        "score",
        pred=pred_path,
        fit=VAL_FIT,
        truth=VAL / "val_truth.csv",
    )
    record = {
        "split_id": split["split_id"],
        "model": name,
        "model_seed": SEED,
        "training_interactions": "validation/val_train.csv",
        "extra_inputs": "none",
        "recall_at_20": score["recall_at_20"],
        "hits": score["hits"],
        "denominator": score["denominator"],
        "notes": f"binary interactions; verify_negative_samples=True; {config}",
    }
    bpr_records[name] = record
    pd.DataFrame(list(bpr_records.values())).to_csv(BPR_EXPERIMENTS, index=False)
    return record


def compare_bpr():
    table = pd.DataFrame([
        *bpr_records.values(),
        {
            "model": "ImplicitALS_k32_reg0.01_alpha40",
            "recall_at_20": 0.034625,
            "hits": 4745,
        },
        {
            "model": "TruncatedSVD_k32_binary",
            "recall_at_20": 0.024584,
            "hits": 3369,
        },
    ])
    table["recall_vs_als"] = (table["recall_at_20"] - 0.034625).round(6)
    table["hits_vs_als"] = table["hits"] - 4745
    display(table.sort_values("hits", ascending=False)[[
        "model", "recall_at_20", "hits", "recall_vs_als", "hits_vs_als"
    ]])


evaluate_bpr(BPR_BASE)
compare_bpr()

In [ ]:
##셀 3 · 반복 횟수와 잠재 차원 비교
def sweep_bpr(current, parameter, values):
    candidates = [current] + [
        {**current, parameter: value}
        for value in values
        if value != current[parameter]
    ]
    scored = [(config, evaluate_bpr(config)) for config in candidates]
    winner, result = max(scored, key=lambda pair: pair[1]["hits"])

    print(
        f"{parameter} → {winner[parameter]} | "
        f"Recall@20 {result['recall_at_20']:.6f} | "
        f"적중 {result['hits']:,}건"
    )
    return winner


bpr_best_iterations = sweep_bpr(BPR_BASE, "iterations", [50, 100, 200])
bpr_best_factors = sweep_bpr(bpr_best_iterations, "factors", [32, 64, 128])

compare_bpr()

In [ ]:
#셀 4 · 학습률과 정규화 비교
bpr_best_lr = sweep_bpr(
    bpr_best_factors, "learning_rate", [0.005, 0.01, 0.05]
)
best_bpr = sweep_bpr(
    bpr_best_lr, "regularization", [0.001, 0.01, 0.05]
)

bpr_winner = evaluate_bpr(best_bpr)
best_bpr_val_pred = WORK / f"{bpr_winner['model']}_seed{SEED}_val.csv"

compare_bpr()
print("BPR-MF 최고 설정 →", best_bpr)
print("팀 공통 실험 기록 →", BPR_EXPERIMENTS)
print("앙상블에 사용할 검증 추천 파일 →", best_bpr_val_pred)

## Logistic MF 기본 실행

In [ ]:
from implicit.cpu.lmf import LogisticMatrixFactorization

LMF_BASE = {
    "factors": 32,
    "iterations": 30,
    "learning_rate": 1.0,
    "regularization": 0.6,
    "neg_prop": 10,
}
LMF_EXPERIMENTS = WORK / "lmf_experiments.csv"
lmf_records = {}


def predict_lmf(fit_data, users_path, output_path, config):
    matrix, row_by_user, item_ids, seen, fallback, catalog = fit_data
    users = read_users(users_path)
    predictions = {}

    model = LogisticMatrixFactorization(
        **config, random_state=SEED, num_threads=THREADS
    )

    with threadpool_limits(limits=1, user_api="blas"):
        model.fit(matrix)

        for start in range(0, len(users), BATCH_SIZE):
            batch = users[start:start + BATCH_SIZE]
            known = [user for user in batch if user in row_by_user]

            if known:
                rows = np.asarray(
                    [row_by_user[user] for user in known], dtype=np.int64
                )
                ranked_cols, scores = model.recommend(
                    rows,
                    matrix[rows],
                    N=min(TOP_K, matrix.shape[1]),
                    filter_already_liked_items=True,
                )
                for user, cols, row_scores in zip(known, ranked_cols, scores):
                    valid = (cols >= 0) & (cols < len(item_ids))
                    valid &= np.isfinite(row_scores)
                    predictions[user] = top20_unseen(
                        user, item_ids[cols[valid]], seen, fallback, catalog
                    )

            for user in batch:
                if user not in row_by_user:
                    predictions[user] = top20_unseen(
                        user, (), seen, fallback, catalog
                    )

    write_predictions(output_path, users, predictions)


def evaluate_lmf(config):
    name = (
        f"LogisticMF_k{config['factors']}_iter{config['iterations']}"
        f"_lr{config['learning_rate']:g}_reg{config['regularization']:g}"
        f"_neg{config['neg_prop']}"
    )
    if name in lmf_records:
        return lmf_records[name]

    pred_path = WORK / f"{name}_seed{SEED}_val.csv"
    print(f"검증 실행 → {name}")
    predict_lmf(VAL_DATA, VAL_USERS, pred_path, config)

    score = run_teamkit(
        "score", pred=pred_path, fit=VAL_FIT, truth=VAL / "val_truth.csv"
    )
    record = {
        "split_id": split["split_id"],
        "model": name,
        "model_seed": SEED,
        "training_interactions": "validation/val_train.csv",
        "extra_inputs": "none",
        "recall_at_20": score["recall_at_20"],
        "hits": score["hits"],
        "denominator": score["denominator"],
        "notes": f"binary interactions; {config}",
    }
    lmf_records[name] = record
    pd.DataFrame(list(lmf_records.values())).to_csv(LMF_EXPERIMENTS, index=False)
    return record


display(pd.DataFrame([evaluate_lmf(LMF_BASE)])[
    ["model", "recall_at_20", "hits"]
])

In [ ]:
# 셀 6 · 잠재 차원과 음성 샘플 비율 비교
def sweep_lmf(current, parameter, values):
    candidates = [current] + [
        {**current, parameter: value}
        for value in values
        if value != current[parameter]
    ]
    return max(candidates, key=lambda config: evaluate_lmf(config)["hits"])


lmf_best_factors = sweep_lmf(LMF_BASE, "factors", [32, 64, 128])
best_lmf = sweep_lmf(lmf_best_factors, "neg_prop", [5, 10, 30])

lmf_winner = evaluate_lmf(best_lmf)
best_lmf_val_pred = WORK / f"{lmf_winner['model']}_seed{SEED}_val.csv"

comparison = pd.DataFrame([
    *lmf_records.values(),
    {"model": "ImplicitALS_k32_reg0.01_alpha40",
     "recall_at_20": 0.034625, "hits": 4745},
    {"model": "TruncatedSVD_k32_binary",
     "recall_at_20": 0.024584, "hits": 3369},
    {"model": "BPRMF_k128_iter100_lr0.01_reg0.01",
     "recall_at_20": 0.013967, "hits": 1914},
    {"model": "Popularity",
     "recall_at_20": 0.022840, "hits": 3130},
])
comparison["hits_vs_als"] = comparison["hits"] - 4745

display(comparison.sort_values("hits", ascending=False)[
    ["model", "recall_at_20", "hits", "hits_vs_als"]
])
print("Logistic MF 최고 설정 →", best_lmf)
print("팀 공통 실험 기록 →", LMF_EXPERIMENTS)
print("앙상블용 검증 추천 파일 →", best_lmf_val_pred)

In [ ]:
## 셀 7 · 음성 샘플 비율 1, 3 비교
lmf_previous_best = best_lmf.copy()
lmf_previous_result = evaluate_lmf(lmf_previous_best)

lmf_best_neg = sweep_lmf(lmf_previous_best, "neg_prop", [1, 3, 5])

neg_results = pd.DataFrame([
    {
        "neg_prop": neg_prop,
        **evaluate_lmf({**lmf_previous_best, "neg_prop": neg_prop}),
    }
    for neg_prop in [1, 3, 5]
])

display(neg_results.sort_values("hits", ascending=False)[
    ["neg_prop", "recall_at_20", "hits"]
])
print("정규화 비교에 사용할 neg_prop →", lmf_best_neg["neg_prop"])

In [ ]:
#셀 8 · 정규화 0.1, 0.3 비교 및 최고 설정 갱신
best_lmf = sweep_lmf(lmf_best_neg, "regularization", [0.1, 0.3, 0.6])

reg_results = pd.DataFrame([
    {
        "regularization": regularization,
        **evaluate_lmf({**lmf_best_neg, "regularization": regularization}),
    }
    for regularization in [0.1, 0.3, 0.6]
])

display(reg_results.sort_values("hits", ascending=False)[
    ["regularization", "recall_at_20", "hits"]
])

lmf_winner = evaluate_lmf(best_lmf)
best_lmf_val_pred = WORK / f"{lmf_winner['model']}_seed{SEED}_val.csv"

display(pd.DataFrame([
    {"stage": "추가 실험 전", **lmf_previous_result},
    {"stage": "추가 실험 후", **lmf_winner},
])[["stage", "model", "recall_at_20", "hits"]])

delta_hits = lmf_winner["hits"] - lmf_previous_result["hits"]
delta_recall = lmf_winner["recall_at_20"] - lmf_previous_result["recall_at_20"]

print(f"기존 최고 대비 → Recall@20 {delta_recall:+.6f} | 적중 {delta_hits:+,}건")
print(f"인기 상품 기준선 대비 → 적중 {lmf_winner['hits'] - 3130:+,}건")
print("LogisticMF 최고 설정 →", best_lmf)
print("팀 공통 실험 기록 →", LMF_EXPERIMENTS)
print("앙상블용 검증 추천 파일 →", best_lmf_val_pred)

---------------------------------------------------------------

### NeuMF 실험 

In [ ]:
#셀 9 · NeuMF 모델 정의
import torch
from torch import nn

NEUMF_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NEUMF_DIM = 32
NEUMF_EPOCHS = 5
NEUMF_LR = 0.001
NEUMF_NEG = 4
NEUMF_BATCH = 4096
NEUMF_USER_BATCH = 32
NEUMF_ITEM_BATCH = 2048

if NEUMF_DEVICE.type == "cpu":
    torch.set_num_threads(THREADS)

print("NeuMF 실행 장치 →", NEUMF_DEVICE)


class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, dim):
        super().__init__()
        self.gmf_user = nn.Embedding(num_users, dim)
        self.gmf_item = nn.Embedding(num_items, dim)
        self.mlp_user = nn.Embedding(num_users, dim)
        self.mlp_item = nn.Embedding(num_items, dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        self.output = nn.Linear(dim + 16, 1)

        for embedding in (
            self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item
        ):
            nn.init.normal_(embedding.weight, std=0.01)

    def forward(self, users, items):
        gmf = self.gmf_user(users) * self.gmf_item(items)
        mlp_input = torch.cat(
            (self.mlp_user(users), self.mlp_item(items)), dim=-1
        )
        mlp = self.mlp(mlp_input)
        return self.output(torch.cat((gmf, mlp), dim=-1)).squeeze(-1)

In [ ]:
#셀 10 · 학습과 전체 상품 Top-20 추천
def sample_neumf_negatives(matrix, rows, rng):
    negatives = rng.integers(matrix.shape[1], size=len(rows))
    observed = np.asarray(matrix[rows, negatives]).ravel() != 0

    while observed.any():
        negatives[observed] = rng.integers(
            matrix.shape[1], size=int(observed.sum())
        )
        observed = np.asarray(matrix[rows, negatives]).ravel() != 0

    return negatives


def fit_neumf(matrix):
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)

    model = NeuMF(*matrix.shape, NEUMF_DIM).to(NEUMF_DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=NEUMF_LR)
    loss_fn = nn.BCEWithLogitsLoss()

    positive_rows, positive_items = matrix.nonzero()
    # 전체 상품을 이미 본 사용자는 미관측 음성 샘플을 만들 수 없음
    eligible = np.diff(matrix.indptr)[positive_rows] < matrix.shape[1]
    positive_rows = positive_rows[eligible]
    positive_items = positive_items[eligible]
    if len(positive_rows) == 0:
        raise ValueError("NeuMF 학습에 사용할 양성·음성 쌍이 없습니다.")

    model.train()
    for epoch in range(1, NEUMF_EPOCHS + 1):
        order = rng.permutation(len(positive_rows))
        total_loss = torch.zeros((), device=NEUMF_DEVICE)
        total_pairs = 0

        for start in range(0, len(order), NEUMF_BATCH):
            indices = order[start:start + NEUMF_BATCH]
            pos_rows = positive_rows[indices]
            pos_items = positive_items[indices]
            neg_rows = np.repeat(pos_rows, NEUMF_NEG)
            neg_items = sample_neumf_negatives(matrix, neg_rows, rng)

            rows = torch.as_tensor(
                np.concatenate((pos_rows, neg_rows)),
                dtype=torch.long, device=NEUMF_DEVICE,
            )
            items = torch.as_tensor(
                np.concatenate((pos_items, neg_items)),
                dtype=torch.long, device=NEUMF_DEVICE,
            )
            labels = torch.zeros(len(rows), device=NEUMF_DEVICE)
            labels[:len(pos_rows)] = 1.0

            optimizer.zero_grad()
            loss = loss_fn(model(rows, items), labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.detach() * len(rows)
            total_pairs += len(rows)

        print(f"Epoch {epoch}/{NEUMF_EPOCHS} · loss {total_loss.item() / total_pairs:.5f}")

    return model


@torch.inference_mode()
def predict_neumf(model, fit_data, users_path, output_path):
    matrix, row_by_user, item_ids, seen, fallback, catalog = fit_data
    users = read_users(users_path)
    predictions = {}
    num_items = matrix.shape[1]
    model.eval()

    for start in range(0, len(users), NEUMF_USER_BATCH):
        batch = users[start:start + NEUMF_USER_BATCH]
        known = [user for user in batch if user in row_by_user]

        for user in batch:
            if user not in row_by_user:
                predictions[user] = top20_unseen(
                    user, (), seen, fallback, catalog
                )

        if not known:
            continue

        rows = np.asarray([row_by_user[user] for user in known])
        row_tensor = torch.as_tensor(rows, dtype=torch.long, device=NEUMF_DEVICE)
        seen_rows, seen_items = matrix[rows].nonzero()
        best_scores = torch.empty((len(rows), 0), device=NEUMF_DEVICE)
        best_items = torch.empty(
            (len(rows), 0), dtype=torch.long, device=NEUMF_DEVICE
        )

        for left in range(0, num_items, NEUMF_ITEM_BATCH):
            right = min(left + NEUMF_ITEM_BATCH, num_items)
            item_block = torch.arange(left, right, device=NEUMF_DEVICE)
            scores = model(
                row_tensor[:, None].expand(-1, len(item_block)),
                item_block[None, :].expand(len(rows), -1),
            )

            in_block = (seen_items >= left) & (seen_items < right)
            mask_rows = torch.as_tensor(
                seen_rows[in_block], dtype=torch.long, device=NEUMF_DEVICE
            )
            mask_cols = torch.as_tensor(
                seen_items[in_block] - left, dtype=torch.long, device=NEUMF_DEVICE
            )
            scores[mask_rows, mask_cols] = -torch.inf

            candidate_scores = torch.cat((best_scores, scores), dim=1)
            candidate_items = torch.cat(
                (best_items, item_block[None, :].expand(len(rows), -1)), dim=1
            )
            best_scores, positions = candidate_scores.topk(
                min(TOP_K, candidate_scores.shape[1]), dim=1
            )
            best_items = candidate_items.gather(1, positions)

        for user, cols, scores in zip(
            known, best_items.cpu().numpy(), best_scores.cpu().numpy()
        ):
            candidates = item_ids[cols[np.isfinite(scores)]]
            predictions[user] = top20_unseen(
                user, candidates, seen, fallback, catalog
            )

    write_predictions(output_path, users, predictions)
    print(f"추천 저장 → {output_path.name} ({len(users):,}명)")

In [ ]:
#셀 11 · 기본 실험 실행 및 비교표 저장
neumf_name = (
    f"NeuMF_k{NEUMF_DIM}_ep{NEUMF_EPOCHS}"
    f"_lr{NEUMF_LR:g}_neg{NEUMF_NEG}"
)
neumf_val_pred = WORK / f"{neumf_name}_seed{SEED}_val.csv"
NEUMF_EXPERIMENTS = WORK / "neumf_experiments.csv"

neumf_model = fit_neumf(VAL_DATA[0])
predict_neumf(neumf_model, VAL_DATA, VAL_USERS, neumf_val_pred)

neumf_score = run_teamkit(
    "score", pred=neumf_val_pred, fit=VAL_FIT, truth=VAL / "val_truth.csv"
)
neumf_record = {
    "split_id": split["split_id"],
    "model": neumf_name,
    "model_seed": SEED,
    "training_interactions": "validation/val_train.csv",
    "extra_inputs": "none",
    "recall_at_20": neumf_score["recall_at_20"],
    "hits": neumf_score["hits"],
    "denominator": neumf_score["denominator"],
    "notes": (
        f"binary interactions; GMF+MLP trained jointly; "
        f"uniform unseen negatives; positive batch={NEUMF_BATCH}; "
        f"device={NEUMF_DEVICE.type}"
    ),
}
pd.DataFrame([neumf_record]).to_csv(NEUMF_EXPERIMENTS, index=False)

neumf_comparison = pd.DataFrame([
    neumf_record,
    {"model": "ImplicitALS", "recall_at_20": 0.034625, "hits": 4745},
    {"model": "TruncatedSVD", "recall_at_20": 0.024584, "hits": 3369},
    {"model": "Popularity", "recall_at_20": 0.022840, "hits": 3130},
    {"model": "LogisticMF", "recall_at_20": 0.018797, "hits": 2576},
    {"model": "BPR-MF", "recall_at_20": 0.013967, "hits": 1914},
])
neumf_comparison["hits_vs_als"] = neumf_comparison["hits"] - 4745

display(neumf_comparison.sort_values("hits", ascending=False)[
    ["model", "recall_at_20", "hits", "hits_vs_als"]
])
print("팀 공통 실험 기록 →", NEUMF_EXPERIMENTS)
print("앙상블용 검증 추천 파일 →", neumf_val_pred)

In [ ]:
####################################
# 중간 저장
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import FileLink, display

working = Path("/kaggle/working")

files = sorted({
    *working.glob("*experiment*.csv"),
    *working.glob("*_val.csv"),
})
if not files:
    raise FileNotFoundError("/kaggle/working에 백업할 실험 CSV가 없어요.")

backup = working / "recommendation_experiments_backup.zip"
with ZipFile(backup, "w", compression=ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, arcname=path.name)

print(f"백업에 포함된 CSV {len(files)}개")
for path in files:
    print(path.name)

display(FileLink("recommendation_experiments_backup.zip",
                 url_prefix="/kaggle/working/"))

------------------------------------------------------------------

## 앙상블 시작

In [ ]:
#셀 12 · 앙상블 입력 불러오기
import re
from zipfile import ZipFile

ALS_VAL_NAME = "ImplicitALS_k32_reg0.01_alpha40_seed42_val.csv"


def find_als_validation_file():
    local = WORK / ALS_VAL_NAME
    if local.is_file():
        return local

    for path in INPUT.rglob(ALS_VAL_NAME):
        return path

    for zip_path in INPUT.rglob("recommendation_experiments_backup.zip"):
        with ZipFile(zip_path) as archive:
            matches = [
                name for name in archive.namelist()
                if Path(name).name == ALS_VAL_NAME
            ]
            if len(matches) == 1:
                local.write_bytes(archive.read(matches[0]))
                return local

    raise FileNotFoundError(
        f"{ALS_VAL_NAME}을 찾을 수 없어요. "
        "GPU 전환 전에 받은 백업 ZIP을 Add Input으로 연결하세요."
    )


def read_ranked_predictions(path, expected_users):
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "user_id" not in table or table["user_id"].duplicated().any():
        raise ValueError(f"사용자 열 또는 중복 사용자 확인 필요 → {path.name}")

    item_columns = [column for column in table.columns if column != "user_id"]
    if len(item_columns) not in (1, TOP_K):
        raise ValueError(f"추천 CSV 열 형식 확인 필요 → {list(table.columns)}")

    rankings = {}
    for user, *values in table[["user_id", *item_columns]].itertuples(
        index=False, name=None
    ):
        if len(values) == 1:
            items = re.split(r"[\s,]+", values[0].strip())
        else:
            items = values

        if len(items) != TOP_K or len(set(items)) != TOP_K or "" in items:
            raise ValueError(f"서로 다른 추천 {TOP_K}개가 필요해요 → {path.name}, {user}")
        rankings[user] = items

    if set(rankings) != set(expected_users):
        raise ValueError(f"검증 사용자 목록과 일치하지 않아요 → {path.name}")

    return rankings


if split["split_id"] != "f0dfad2f4431a876":
    raise ValueError(f"팀 공통 분할과 달라요 → {split['split_id']}")

ensemble_users = read_users(VAL_USERS)
als_val_pred = find_als_validation_file()

als_ranks = read_ranked_predictions(als_val_pred, ensemble_users)
neumf_ranks = read_ranked_predictions(neumf_val_pred, ensemble_users)

print("ALS 검증 추천 →", als_val_pred)
print("NeuMF 검증 추천 →", neumf_val_pred)
print(f"검증 사용자 → {len(ensemble_users):,}명")

In [ ]:
#셀 13 · 가중 순위 결합 및 공통 채점
RRF_K = 10
NEUMF_WEIGHTS = [0.4, 0.6, 0.8]

_, _, _, ensemble_seen, ensemble_fallback, ensemble_catalog = VAL_DATA


def score_validation_prediction(path):
    return run_teamkit(
        "score", pred=path, fit=VAL_FIT, truth=VAL / "val_truth.csv"
    )


# 백업에서 복원한 ALS 파일이 기존 최고 설정의 결과인지 재채점
als_baseline = score_validation_prediction(als_val_pred)
if als_baseline["hits"] != 4745:
    raise ValueError(
        f"복원한 ALS 파일의 적중 수가 기존 4,745건과 달라요 → "
        f"{als_baseline['hits']:,}건"
    )

ensemble_records = [{
    "model": "ImplicitALS",
    "neumf_weight": 0.0,
    "recall_at_20": als_baseline["recall_at_20"],
    "hits": als_baseline["hits"],
    "prediction_file": als_val_pred.name,
}]

for weight in NEUMF_WEIGHTS:
    predictions = {}

    for user in ensemble_users:
        scores = {}
        for model_weight, items in ((1.0, als_ranks[user]), (weight, neumf_ranks[user])):
            for rank, item in enumerate(items, start=1):
                scores[item] = scores.get(item, 0.0) + model_weight / (RRF_K + rank)

        # 결합 점수가 같으면 ALS에서 더 높았던 상품을 우선함
        als_positions = {item: rank for rank, item in enumerate(als_ranks[user])}
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    pred_path = WORK / f"ensemble_als_neumf_rrf{RRF_K}_w{weight:g}_val.csv"
    write_predictions(pred_path, ensemble_users, predictions)
    result = score_validation_prediction(pred_path)

    ensemble_records.append({
        "model": "ALS + NeuMF",
        "neumf_weight": weight,
        "recall_at_20": result["recall_at_20"],
        "hits": result["hits"],
        "prediction_file": pred_path.name,
    })

ensemble_results = pd.DataFrame(ensemble_records)
ensemble_results["recall_vs_als"] = (
    ensemble_results["recall_at_20"] - als_baseline["recall_at_20"]
).round(6)
ensemble_results["hits_vs_als"] = ensemble_results["hits"] - als_baseline["hits"]
ensemble_results = ensemble_results.sort_values("hits", ascending=False, kind="stable")

ensemble_results.to_csv(WORK / "als_neumf_ensemble_experiments.csv", index=False)
display(ensemble_results)

In [ ]:
#셀 14 · SVD 검증 추천 파일 연결 및 확인
SVD_VAL_NAME = "TruncatedSVD_k32_binary_seed42_val.csv"
local_svd = WORK / SVD_VAL_NAME

if local_svd.is_file():
    svd_val_pred = local_svd
else:
    svd_matches = sorted(INPUT.rglob(SVD_VAL_NAME))
    if len(svd_matches) != 1:
        raise FileNotFoundError(
            f"{SVD_VAL_NAME}을 하나 연결해 주세요. "
            f"현재 일치하는 Input 파일은 {len(svd_matches)}개예요."
        )
    svd_val_pred = svd_matches[0]

svd_ranks = read_ranked_predictions(svd_val_pred, ensemble_users)
svd_baseline = score_validation_prediction(svd_val_pred)

if svd_baseline["hits"] != 3369:
    raise ValueError(
        f"SVD 재채점 결과가 기존 3,369건과 달라요. "
        f"현재 적중 수는 {svd_baseline['hits']:,}건이에요."
    )

print("SVD 검증 추천 →", svd_val_pred)
print(f"SVD 단독 · Recall@20 {svd_baseline['recall_at_20']:.6f} · 적중 3,369건")

In [ ]:
##추가 
#SVD 실행 셀 추가
from sklearn.decomposition import TruncatedSVD
N_COMPONENTS = 64
SEED = 42
N_ITER = 7
BATCH_SIZE = 128
TOP_K = 20


def predict_svd(
    fit_path,
    users_path,
    output_path,
    *,
    n_components=N_COMPONENTS,
    weighted=False,
):
    fit = pd.read_csv(fit_path, usecols=["user_id", "item_id"])
    user_rows, user_ids = pd.factorize(fit["user_id"], sort=True)
    item_cols, item_ids = pd.factorize(fit["item_id"], sort=True)

    # 사용자–상품 상호작용을 희소 행렬로 구성
    matrix = csr_matrix(
        (np.ones(len(fit), dtype=np.float32), (user_rows, item_cols)),
        shape=(len(user_ids), len(item_ids)),
    )
    matrix.data.fill(1.0)  # 같은 사용자–상품 쌍이 반복돼도 1로 처리
    matrix.sort_indices()

    if weighted:
        # 상품별로 상호작용한 고유 사용자 수를 계산
        item_user_counts = np.bincount(
            matrix.indices,
            minlength=matrix.shape[1],
        )
        item_weights = (1.0 / np.sqrt(item_user_counts)).astype(np.float32)

        # 상호작용이 있는 칸에 1 / sqrt(상품별 사용자 수)를 적용
        matrix.data *= item_weights[matrix.indices]

    model = TruncatedSVD(
        n_components=n_components,
        n_iter=N_ITER,
        random_state=SEED,
    )

    seen, popularity = read_interactions(fit_path)
    catalog = set(popularity)
    fallback = sorted(catalog, key=lambda item: (-popularity[item], int(item)))
    users = read_users(users_path)

    row_by_user = {str(user): row for row, user in enumerate(user_ids)}
    item_names = np.asarray(item_ids.astype(str))
    predictions = {}

    with threadpool_limits(limits=2, user_api="blas"):
        user_factors = model.fit_transform(matrix)

        for start in range(0, len(users), BATCH_SIZE):
            batch = users[start:start + BATCH_SIZE]
            known = [
                (user, row_by_user[user])
                for user in batch
                if user in row_by_user
            ]

            if known:
                scores = user_factors[[row for _, row in known]] @ model.components_

                for (user, row), score in zip(known, scores):
                    # 학습 데이터에서 이미 본 상품을 추천 후보에서 제외
                    row_start = matrix.indptr[row]
                    row_end = matrix.indptr[row + 1]
                    seen_cols = matrix.indices[row_start:row_end]
                    score[seen_cols] = -np.inf

                    top = np.argpartition(score, -TOP_K)[-TOP_K:]
                    top = top[np.lexsort((top, -score[top]))]
                    candidates = item_names[top[np.isfinite(score[top])]]

                    predictions[user] = top20_unseen(
                        user, candidates, seen, fallback, catalog
                    )

            # 학습 기록이 없는 사용자에게는 인기 상품으로 추천
            for user in batch:
                if user not in row_by_user:
                    predictions[user] = top20_unseen(
                        user, (), seen, fallback, catalog
                    )

    write_predictions(output_path, users, predictions)
    print(f"사용자 {len(users):,}명의 추천 저장 → {output_path}")

In [ ]:
#위에서 이어서
svd_val_path = WORK / "TruncatedSVD_k32_binary_seed42_val.csv"

predict_svd(
    fit_path=VAL_FIT,
    users_path=VAL_USERS,
    output_path=svd_val_path,
    n_components=32,
    weighted=False,
)

print("SVD 검증 추천 저장 →", svd_val_path)

In [ ]:
#셀 15 · ALS + SVD 가중치 비교
SVD_WEIGHTS = [0.4, 0.6, 0.8]
svd_ensemble_records = []

for weight in SVD_WEIGHTS:
    predictions = {}

    for user in ensemble_users:
        scores = {}
        for model_weight, items in (
            (1.0, als_ranks[user]),
            (weight, svd_ranks[user]),
        ):
            for rank, item in enumerate(items, start=1):
                scores[item] = (
                    scores.get(item, 0.0) + model_weight / (RRF_K + rank)
                )

        als_positions = {
            item: rank for rank, item in enumerate(als_ranks[user])
        }
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    pred_path = WORK / f"ensemble_als_svd_rrf{RRF_K}_w{weight:g}_val.csv"
    write_predictions(pred_path, ensemble_users, predictions)
    result = score_validation_prediction(pred_path)

    svd_ensemble_records.append({
        "model": "ALS + SVD",
        "secondary_weight": weight,
        "recall_at_20": result["recall_at_20"],
        "hits": result["hits"],
        "prediction_file": pred_path.name,
    })

svd_ensemble_results = pd.DataFrame(svd_ensemble_records)
ensemble_comparison = pd.concat([
    ensemble_results.rename(columns={"neumf_weight": "secondary_weight"}),
    svd_ensemble_results,
], ignore_index=True)

ensemble_comparison["recall_vs_als"] = (
    ensemble_comparison["recall_at_20"] - als_baseline["recall_at_20"]
).round(6)
ensemble_comparison["hits_vs_als"] = (
    ensemble_comparison["hits"] - als_baseline["hits"]
)
neumf_best_hits = ensemble_results.loc[
    ensemble_results["model"] == "ALS + NeuMF", "hits"
].max()
ensemble_comparison["hits_vs_best_als_neumf"] = (
    ensemble_comparison["hits"] - neumf_best_hits
)

ensemble_comparison = ensemble_comparison.sort_values(
    "hits", ascending=False, kind="stable"
)
comparison_path = WORK / "als_neumf_svd_ensemble_comparison.csv"
ensemble_comparison.to_csv(comparison_path, index=False)

display(ensemble_comparison[[
    "model", "secondary_weight", "recall_at_20", "hits",
    "hits_vs_als", "hits_vs_best_als_neumf",
]])
print("비교 결과 저장 →", comparison_path)

In [ ]:
#셀 16 · NeuMF 가중치 0.3, 0.5 비교
def evaluate_als_neumf_weight(weight):
    predictions = {}

    for user in ensemble_users:
        als_items = als_ranks[user]
        scores = {
            item: 1.0 / (RRF_K + rank)
            for rank, item in enumerate(als_items, start=1)
        }

        for rank, item in enumerate(neumf_ranks[user], start=1):
            scores[item] = scores.get(item, 0.0) + weight / (RRF_K + rank)

        # 셀 13과 같은 동점 처리 규칙
        als_positions = {item: rank for rank, item in enumerate(als_items)}
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    pred_path = WORK / f"ensemble_als_neumf_rrf{RRF_K}_w{weight:g}_val.csv"
    write_predictions(pred_path, ensemble_users, predictions)
    result = score_validation_prediction(pred_path)

    return {
        "model": "ALS + NeuMF",
        "neumf_weight": weight,
        "recall_at_20": result["recall_at_20"],
        "hits": result["hits"],
        "prediction_file": pred_path.name,
    }


# 기존 결과를 먼저 두고 안정 정렬해 동점이면 기존 최고 설정을 유지
refined_results = pd.concat([
    ensemble_results,
    pd.DataFrame([evaluate_als_neumf_weight(w) for w in (0.3, 0.5)]),
], ignore_index=True)

previous_best = ensemble_results.loc[
    ensemble_results["model"] == "ALS + NeuMF"
].iloc[0]

refined_results["recall_vs_als"] = (
    refined_results["recall_at_20"] - als_baseline["recall_at_20"]
).round(6)
refined_results["hits_vs_als"] = refined_results["hits"] - als_baseline["hits"]
refined_results["hits_vs_previous_best"] = (
    refined_results["hits"] - previous_best["hits"]
)
refined_results = refined_results.sort_values(
    "hits", ascending=False, kind="stable"
).reset_index(drop=True)

best_ensemble = refined_results.iloc[0]
best_ensemble_val_pred = WORK / best_ensemble["prediction_file"]

refined_results.to_csv(
    WORK / "als_neumf_ensemble_refined_experiments.csv", index=False
)

display(refined_results[[
    "model", "neumf_weight", "recall_at_20", "hits",
    "hits_vs_als", "hits_vs_previous_best",
]])
print(f"최고 NeuMF 가중치 → {best_ensemble['neumf_weight']:g}")
print("최고 조합 검증 추천 파일 →", best_ensemble_val_pred)

In [ ]:
#셀 17 · ALS + NeuMF + SVD 검증
base_hits = int(best_ensemble["hits"])
neumf_weight = float(best_ensemble["neumf_weight"])

three_model_records = [{
    "model": "ALS + NeuMF",
    "neumf_weight": neumf_weight,
    "svd_weight": 0.0,
    "recall_at_20": best_ensemble["recall_at_20"],
    "hits": base_hits,
    "prediction_file": best_ensemble["prediction_file"],
}]

for svd_weight in (0.1, 0.2, 0.3):
    predictions = {}

    for user in ensemble_users:
        als_items = als_ranks[user]
        scores = {}

        for weight, items in (
            (1.0, als_items),
            (neumf_weight, neumf_ranks[user]),
            (svd_weight, svd_ranks[user]),
        ):
            for rank, item in enumerate(items, start=1):
                scores[item] = scores.get(item, 0.0) + weight / (RRF_K + rank)

        als_positions = {item: rank for rank, item in enumerate(als_items)}
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    pred_path = WORK / (
        f"ensemble_als_neumf_svd_rrf{RRF_K}"
        f"_neu{neumf_weight:g}_svd{svd_weight:g}_val.csv"
    )
    write_predictions(pred_path, ensemble_users, predictions)
    result = score_validation_prediction(pred_path)

    three_model_records.append({
        "model": "ALS + NeuMF + SVD",
        "neumf_weight": neumf_weight,
        "svd_weight": svd_weight,
        "recall_at_20": result["recall_at_20"],
        "hits": result["hits"],
        "prediction_file": pred_path.name,
    })
    print(f"SVD 가중치 {svd_weight:g} · 적중 {result['hits']:,}건")

three_model_results = pd.DataFrame(three_model_records)
three_model_results["hits_vs_als_neumf"] = three_model_results["hits"] - base_hits
three_model_results = three_model_results.sort_values(
    "hits", ascending=False, kind="stable"
).reset_index(drop=True)

three_model_results.to_csv(
    WORK / "als_neumf_svd_three_model_experiments.csv", index=False
)

display(three_model_results[[
    "model", "neumf_weight", "svd_weight",
    "recall_at_20", "hits", "hits_vs_als_neumf",
]])
print("검증 최고 추천 파일 →", WORK / three_model_results.iloc[0]["prediction_file"])

In [ ]:
%pip install --only-binary=:all: implicit==0.7.3

from importlib.metadata import version
from implicit.cpu.als import AlternatingLeastSquares

print(f"implicit {version('implicit')} · CPU ALS 준비 완료")

In [ ]:
#추가 셀 A · ALS와 SVD의 Top-100 후보 생성
from importlib.metadata import version

from implicit.cpu.als import AlternatingLeastSquares
from sklearn.decomposition import TruncatedSVD

DEPTH = 100

matrix, row_by_user, item_ids, seen, fallback, catalog = VAL_DATA
print("ALS 라이브러리 버전 →", version("implicit"))

depth_als = AlternatingLeastSquares(
    factors=32,
    regularization=0.01,
    alpha=40.0,
    iterations=20,
    random_state=42,
    num_threads=THREADS,
)
depth_svd = TruncatedSVD(n_components=32, n_iter=7, random_state=42)

als_deep_ranks = {}
svd_deep_ranks = {}

with threadpool_limits(limits=1, user_api="blas"):
    depth_als.fit(matrix)

# 기존 SVD 실험과 동일한 숫자 ID 정렬 순서로 행렬을 구성함
fit = pd.read_csv(VAL_FIT, usecols=["user_id", "item_id"])
svd_rows, svd_users = pd.factorize(fit["user_id"], sort=True)
svd_cols, svd_items = pd.factorize(fit["item_id"], sort=True)
svd_matrix = csr_matrix(
    (np.ones(len(fit), dtype=np.float32), (svd_rows, svd_cols)),
    shape=(len(svd_users), len(svd_items)),
)
svd_matrix.data.fill(1.0)
svd_matrix.sort_indices()
svd_row_by_user = {str(user): row for row, user in enumerate(svd_users)}
svd_item_ids = np.asarray(svd_items.astype(str))

with threadpool_limits(limits=2, user_api="blas"):
    svd_factors = depth_svd.fit_transform(svd_matrix)


def fill_candidate_ranks(user, candidates, depth):
    blocked = set(seen.get(user, ()))
    ranked = []

    for source in (candidates, fallback):
        for item in source:
            if item in blocked or item not in catalog:
                continue
            ranked.append(item)
            blocked.add(item)
            if len(ranked) == depth:
                return ranked

    return ranked


for start in range(0, len(ensemble_users), 128):
    batch = ensemble_users[start:start + 128]
    known = [user for user in batch if user in row_by_user]

    if known:
        rows = np.asarray([row_by_user[user] for user in known])
        with threadpool_limits(limits=1, user_api="blas"):
            als_cols, als_scores = depth_als.recommend(
                rows,
                matrix[rows],
                N=min(DEPTH, matrix.shape[1]),
                filter_already_liked_items=True,
            )

        svd_batch_rows = [svd_row_by_user[user] for user in known]
        with threadpool_limits(limits=2, user_api="blas"):
            svd_scores = svd_factors[svd_batch_rows] @ depth_svd.components_

        for user, cols, scores, svd_row, svd_score in zip(
            known, als_cols, als_scores, svd_batch_rows, svd_scores
        ):
            valid = (cols >= 0) & (cols < len(item_ids)) & np.isfinite(scores)
            als_deep_ranks[user] = fill_candidate_ranks(
                user, item_ids[cols[valid]], DEPTH
            )

            left, right = svd_matrix.indptr[svd_row:svd_row + 2]
            svd_score[svd_matrix.indices[left:right]] = -np.inf
            take = min(DEPTH, len(svd_score))
            top = np.argpartition(svd_score, -take)[-take:]
            top = top[np.lexsort((top, -svd_score[top]))]
            svd_deep_ranks[user] = fill_candidate_ranks(
                user, svd_item_ids[top[np.isfinite(svd_score[top])]], DEPTH
            )

    for user in batch:
        if user not in row_by_user:
            cold_ranks = fill_candidate_ranks(user, (), DEPTH)
            als_deep_ranks[user] = cold_ranks
            svd_deep_ranks[user] = cold_ranks

print(f"ALS 후보 생성 → {len(als_deep_ranks):,}명")
print(f"SVD 후보 생성 → {len(svd_deep_ranks):,}명")

In [ ]:
#추가 셀 B · 후보 깊이만 바꿔 검증
def evaluate_candidate_depth(als_depth, svd_depth):
    predictions = {}

    for user in ensemble_users:
        als_items = als_deep_ranks[user][:als_depth]
        scores = {}

        for weight, items in (
            (1.0, als_items),
            (0.4, neumf_ranks[user]),
            (0.2, svd_deep_ranks[user][:svd_depth]),
        ):
            for rank, item in enumerate(items, start=1):
                scores[item] = scores.get(item, 0.0) + weight / (10 + rank)

        als_positions = {item: index for index, item in enumerate(als_items)}
        candidates = sorted(
            scores,
            key=lambda item: (
                -scores[item],
                als_positions.get(item, len(als_items)),
                item,
            ),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    path = WORK / f"ensemble_depth_als{als_depth}_neu20_svd{svd_depth}_val.csv"
    write_predictions(path, ensemble_users, predictions)
    score = score_validation_prediction(path)

    return {
        "als_depth": als_depth,
        "svd_depth": svd_depth,
        "recall_at_20": score["recall_at_20"],
        "hits": score["hits"],
        "prediction_file": path.name,
    }


depth_results = pd.DataFrame([
    evaluate_candidate_depth(20, 20),
    evaluate_candidate_depth(100, 20),
    evaluate_candidate_depth(100, 100),
])

control_hits = int(depth_results.iloc[0]["hits"])
depth_results["hits_vs_same_fit_top20"] = depth_results["hits"] - control_hits
depth_results["hits_vs_previous_best"] = depth_results["hits"] - 5195

depth_results = depth_results.sort_values(
    "hits", ascending=False, kind="stable"
).reset_index(drop=True)

depth_results.to_csv(WORK / "ensemble_candidate_depth_experiments.csv", index=False)
display(depth_results)

In [ ]:
#셀 C · ALS alpha=80, 160 검증
def score_alpha_ensemble(als_rankings, alpha):
    predictions = {}

    for user in ensemble_users:
        als_items = als_rankings[user]
        scores = {}

        for weight, items in (
            (1.0, als_items),
            (0.4, neumf_ranks[user]),
            (0.2, svd_deep_ranks[user][:TOP_K]),
        ):
            for rank, item in enumerate(items, start=1):
                scores[item] = scores.get(item, 0.0) + weight / (10 + rank)

        als_positions = {item: idx for idx, item in enumerate(als_items)}
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, ensemble_seen, ensemble_fallback, ensemble_catalog
        )

    path = WORK / f"ensemble_rrf10_als_alpha{alpha}_neu0.4_svd0.2_val.csv"
    write_predictions(path, ensemble_users, predictions)
    return path, score_validation_prediction(path)


alpha_records = []

for alpha in (40, 80, 160):
    print(f"ALS alpha={alpha} 검증 시작", flush=True)
    als_path = WORK / f"als073_k32_reg0.01_alpha{alpha}_seed42_val.csv"

    if alpha == 40:
        write_predictions(
            als_path,
            ensemble_users,
            {user: als_deep_ranks[user][:TOP_K] for user in ensemble_users},
        )
    else:
        predict_als(
            VAL_DATA,
            VAL_USERS,
            als_path,
            {"factors": 32, "regularization": 0.01, "alpha": float(alpha)},
        )

    als_score = score_validation_prediction(als_path)
    als_rankings = read_ranked_predictions(als_path, ensemble_users)
    ensemble_path, ensemble_score = score_alpha_ensemble(als_rankings, alpha)

    alpha_records.append({
        "alpha": alpha,
        "als_recall_at_20": als_score["recall_at_20"],
        "als_hits": als_score["hits"],
        "ensemble_recall_at_20": ensemble_score["recall_at_20"],
        "ensemble_hits": ensemble_score["hits"],
        "hits_vs_previous_best": ensemble_score["hits"] - 5195,
        "implicit_version": version("implicit"),
        "prediction_file": ensemble_path.name,
    })
    print(f"ALS {als_score['hits']:,}건 · 앙상블 {ensemble_score['hits']:,}건")

alpha_results = pd.DataFrame(alpha_records).sort_values(
    "ensemble_hits", ascending=False, kind="stable"
)
alpha_results.to_csv(WORK / "als_alpha_ensemble_experiments.csv", index=False)
display(alpha_results)

In [ ]:
#마지막...발악....
#셀 F · NeuMF 10 epoch 학습 및 검증 추천 저장
assert NEUMF_DEVICE.type == "cuda", "GPU 세션에서 실행하세요."
assert (NEUMF_DIM, NEUMF_LR, NEUMF_NEG, SEED) == (32, 0.001, 4, 42)

neu10_path = WORK / "NeuMF_k32_ep10_lr0.001_neg4_seed42_val.csv"

if not neu10_path.is_file():
    original_epochs = NEUMF_EPOCHS
    try:
        NEUMF_EPOCHS = 10
        neu10_model = fit_neumf(VAL_DATA[0])

        print("NeuMF 학습 완료 · 전체 상품 점수 계산 시작", flush=True)
        temp_path = neu10_path.with_suffix(".tmp.csv")
        predict_neumf(neu10_model, VAL_DATA, VAL_USERS, temp_path)
        read_ranked_predictions(temp_path, ensemble_users)
        temp_path.replace(neu10_path)
    finally:
        NEUMF_EPOCHS = original_epochs

neu10_ranks = read_ranked_predictions(neu10_path, ensemble_users)
neu10_score = score_validation_prediction(neu10_path)

print(
    f"NeuMF 10 epoch 단독 → Recall@20 {neu10_score['recall_at_20']:.6f}"
    f" · 적중 {neu10_score['hits']:,}건"
)

In [ ]:
#셀 G · 현재 최고와 NeuMF 교체·결합 비교
last_als_ranks = read_ranked_predictions(
    WORK / "als073_k32_reg0.01_alpha40_seed42_val.csv",
    ensemble_users,
)
last_svd_ranks = {
    user: svd_deep_ranks[user][:TOP_K] for user in ensemble_users
}

LAST_CONFIGS = {
    "neu5": {"als": 1.0, "neu5": 0.4, "svd": 0.2},
    "neu10": {"als": 1.0, "neu10": 0.4, "svd": 0.2},
    "neu5_neu10": {"als": 1.0, "neu5": 0.2, "neu10": 0.2, "svd": 0.2},
}
last_rankings = {
    "als": last_als_ranks,
    "svd": last_svd_ranks,
    "neu5": neumf_ranks,
    "neu10": neu10_ranks,
}


def combine_epoch_rankings(users, rankings, weights, fit_data):
    _, _, _, seen, fallback, catalog = fit_data
    predictions = {}

    for user in users:
        scores = {}
        for model, weight in weights.items():
            for rank, item in enumerate(rankings[model][user], start=1):
                scores[item] = scores.get(item, 0.0) + weight / (10 + rank)

        als_positions = {
            item: index for index, item in enumerate(rankings["als"][user])
        }
        candidates = sorted(
            scores,
            key=lambda item: (-scores[item], als_positions.get(item, TOP_K), item),
        )
        predictions[user] = top20_unseen(
            user, candidates, seen, fallback, catalog
        )

    return predictions


last_records = []

for label, weights in LAST_CONFIGS.items():
    path = WORK / f"ensemble_als_svd_{label}_rrf10_val.csv"
    predictions = combine_epoch_rankings(
        ensemble_users, last_rankings, weights, VAL_DATA
    )
    write_predictions(path, ensemble_users, predictions)
    score = score_validation_prediction(path)

    last_records.append({
        "config": label,
        "recall_at_20": score["recall_at_20"],
        "hits": score["hits"],
        "hits_vs_previous_best": score["hits"] - 5195,
        "prediction_file": path.name,
    })
    print(f"{label} → {score['hits']:,}건", flush=True)

last_results = pd.DataFrame(last_records)
control_hits = int(last_results.loc[last_results["config"].eq("neu5"), "hits"].iloc[0])
if control_hits != 5195:
    raise ValueError(
        f"기존 조합이 5,195건으로 재현되지 않았어요. 현재 {control_hits:,}건"
    )

last_results = last_results.sort_values(
    "hits", ascending=False, kind="stable"
).reset_index(drop=True)
last_results.to_csv(WORK / "neumf_epoch_ensemble_experiments.csv", index=False)

winner_label = last_results.iloc[0]["config"]
winner_weights = LAST_CONFIGS[winner_label].copy()

display(last_results)
print("최고 조합 →", winner_weights)

-------------최종제출

In [ ]:
#제출 셀 1 · 전체 학습 및 모델별 테스트 추천 생성
from importlib.metadata import version
from implicit.cpu.als import AlternatingLeastSquares

assert version("implicit") == "0.7.3", "검증에 사용한 implicit 0.7.3이 필요해요."
assert NEUMF_DEVICE.type == "cuda", "NeuMF가 GPU에 연결됐는지 확인하세요."
assert (SEED, ITERATIONS, TOP_K) == (42, 20, 20)
assert (NEUMF_DIM, NEUMF_LR, NEUMF_NEG) == (32, 0.001, 4)
assert N_ITER == 7, "SVD의 n_iter는 검증 때와 같은 7이어야 해요."

submission_weights = {"als": 1.0, "neu5": 0.4, "svd": 0.2}
submission_als_config = {"factors": 32, "regularization": 0.01, "alpha": 40.0}

submission_dir = WORK / "final_als_neu5_svd"
submission_dir.mkdir(exist_ok=True)
submission_test_paths = {
    model: submission_dir / f"{model}_test.csv"
    for model in submission_weights
}

submission_fit_data = load_fit_data(TRAIN)

print("1/3 · ALS 전체 학습 및 테스트 추천", flush=True)
predict_als(
    submission_fit_data, TEST, submission_test_paths["als"], submission_als_config
)

print("2/3 · SVD 전체 학습 및 테스트 추천", flush=True)
predict_svd(
    fit_path=TRAIN,
    users_path=TEST,
    output_path=submission_test_paths["svd"],
    n_components=32,
    weighted=False,
)

print("3/3 · NeuMF 5 epoch 전체 학습", flush=True)
saved_epochs = NEUMF_EPOCHS
try:
    NEUMF_EPOCHS = 5
    submission_neumf = fit_neumf(submission_fit_data[0])
finally:
    NEUMF_EPOCHS = saved_epochs

print("NeuMF 학습 완료 · 테스트 전체 상품 점수 계산 시작", flush=True)
predict_neumf(
    submission_neumf, submission_fit_data, TEST, submission_test_paths["neu5"]
)

In [ ]:
#셀 2 · 추천 결합, 검사, 다운로드
from IPython.display import FileLink, display

submission_users = read_users(TEST)
submission_rankings = {
    model: read_ranked_predictions(path, submission_users)
    for model, path in submission_test_paths.items()
}

submission_predictions = combine_epoch_rankings(
    submission_users,
    submission_rankings,
    submission_weights,
    submission_fit_data,
)

candidate_path = submission_dir / "submission_candidate.csv"
submission_path = WORK / "submission.csv"

write_predictions(candidate_path, submission_users, submission_predictions)
check_result = run_teamkit("check", pred=candidate_path, train=TRAIN, test=TEST)
print(json.dumps(check_result, indent=2, ensure_ascii=False))

candidate_path.replace(submission_path)
print(f"제출 파일 저장 완료 → {submission_path} ({len(submission_users):,}명)")
display(FileLink("submission.csv", url_prefix="/kaggle/working/"))

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, display

working = Path("/kaggle/working")
zip_path = working / "validation_top20_all_experiments.zip"

validation_files = sorted(
    path for path in working.rglob("*.csv")
    if path.is_file()
    and (
        "validation_top20" in path.stem.lower()
        or path.name.lower().endswith("_val.csv")
    )
    and not path.name.lower().endswith(".tmp.csv")
)

if not validation_files:
    raise FileNotFoundError(
        "/kaggle/working에서 검증 추천 CSV를 찾지 못했어요. "
        "실험 셀에서 CSV 저장이 완료됐는지 확인하세요."
    )

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in validation_files:
        archive.write(path, arcname=path.relative_to(working))

print(f"ZIP 생성 완료 · 검증 추천 CSV {len(validation_files):,}개")
for path in validation_files:
    print(f"  {path.relative_to(working)}")

display(FileLink(zip_path.name, url_prefix="/kaggle/working/"))